In [349]:
import json
import pathlib
import json
import re
import asyncio
import datetime
from typing import Any
from functools import lru_cache

In [350]:
base_dir = pathlib.Path('.').absolute()

In [351]:
corrected_path = base_dir / 'corrected'

In [352]:
countries_path = base_dir.parent / 'geo' / 'countries.json'
countries_by_continent_path = base_dir.parent / 'geo' / 'countries_by_continent.json'

In [353]:
with countries_path.open() as f:
    countries: list[dict[str, Any]] = json.load(f)

with countries_by_continent_path.open() as f:
    countries_by_continent: list[dict[str, str]] = json.load(f)

In [354]:
@lru_cache(maxsize=None)
def get_country(country_code: str) -> list[dict[str, Any]]:
    return list(
        filter(
            lambda x: x['code-alpha-3'].lower() == country_code.lower(),
            countries
        )
    )

In [355]:
@lru_cache(maxsize=None)
def get_country_by_continent(country_code: str) -> list[dict[str, Any]]:
    return list(
        filter(
            lambda x: x['code-alpha-3'].lower() == country_code.lower(),
            countries_by_continent
        )
    )

In [356]:
data_dir = base_dir / 'data'

In [357]:
files = list(data_dir.glob('*.json'))

## Cleaning

In [358]:
def clean_location(data: dict):
    """Fixes the location field by removing the date and replacing 
    multiple spaces with a single space."""
    clean_location = re.sub(r'\s+', ' ', data['location'])
    data['location'] = clean_location

In [359]:
def get_location(data: dict[str, str]):
    """Extracts the location from the location field by removing the date.
    Should be applied once the location field has been cleaned."""
    location = data['location']
    location = location.strip().removesuffix(',')

    def parse_single(value: str):
        city, _ = value.split('•')
        return city.strip()
    
    city = None
    state = None

    if 'UNITED STATES' in location:
        # NEW YORK • NY, UNITED STATES -> [NEW YORK • NY, UNITED STATES]
        if location.endswith('UNITED STATES'):
            tokens = location.split(',')
            if len(tokens) == 1:
                # Some locations do not provide a state e.g. ['WASHINGTON DC • UNITED STATES']
                city = parse_single(tokens[0])
            else:
                lhv, _ = tokens
                city, state = lhv.split('•')
        else:
            tokens = location.split(',')
            if len(tokens) == 1:
                # Some locations do not provide a state e.g. ['WASHINGTON DC • UNITED STATES']
                city = parse_single(tokens[0])
            else:
                # INDIAN WELLS • UNITED STATES, CA -> [INDIAN WELLS • UNITED STATES, CA]
                lhv, state = location.split(',')
                city, _ = lhv.split('•')
        country = 'United States of America'
    else:
        tokens = location.split('•')
        if len(tokens) == 1:
            country = tokens[0]
        else:
            city, country = tokens

    data['city'] = city.strip() if city is not None else None
    data['country'] = country.strip() if country is not None else None
    data['state'] = state

    if state is not None:
        data['state'] = state.strip()

    data.pop('location')

## Dates

In [360]:
def start_end_date(data: dict):
    """Extract start and end date from location field and add them as 
    separate fields in the data dictionary."""
    date_regex = r'\d+ - \d+ .*'

    location = data['location']
    result = re.search(date_regex, location)

    if location == '' or location is None:
        data['start_date'] = None
        data['end_date'] = None
        data['year'] = None
        data['month'] = None
        return

    if result is not None:
        # Resolve this format: 22 - 28 Jun 2025
        date_tokens = result.group().split('-')
        tokens = list(map(lambda x: x.strip(), date_tokens))

        d, m, y = tokens[1].split(' ')
        month = datetime.datetime.strptime(m, '%b').month
        
        start_date = datetime.datetime(int(y), month, int(tokens[0]))
        end_date = datetime.datetime(int(y), month, int(d))
        # Remove the date from the location field
        data['location'] = re.sub(date_regex, '', location)
    else:
        # Result this format: NEW YORK • USA, 24 Aug - 7 Sep 2025
        result = re.search(r'\d+ \w+ - \d+ .*', location)

        if result is None:
            data['start_date'] = None
            data['end_date'] = None
            data['year'] = None
            data['month'] = None
            return

        date_tokens = result.group().split('-')
        tokens = list(map(lambda x: x.strip(), date_tokens))

        year = re.search(r'\d{4}$', tokens[1])
        if year is None:
            data['start_date'] = None
            data['end_date'] = None
            data['year'] = None
            data['month'] = None
            return
        else:
            tokens[0] = tokens[0] + ' ' + year.group()

        start_date = datetime.datetime.strptime(tokens[0], '%d %b %Y')
        end_date = datetime.datetime.strptime(tokens[1], '%d %b %Y')

        # Remove the date from the location field
        data['location'] = re.sub(r'\d+ \w+ - \d+ .*', '', location)

    
    data['start_date'] = start_date.date().isoformat()
    data['end_date'] = end_date.date().isoformat()
    data['year'] = start_date.year
    data['month'] = start_date.month

## Score

In [361]:
def correct_score(value: str):
    """Corrects the score by replacing multiple spaces with a single space."""
    return re.sub(r'\s+', ' ', value)

In [362]:
def correct_data(data: dict[str, str]) -> list:
    fixes = [
       clean_location,
       start_end_date,
       get_location
    ]

    for fix in fixes:
        fix(data)

    for item in data['matches']:
        item['score'] = correct_score(item['score'])

        # Correct the seed field by extracting the number 
        # from the string and converting it to an integer
        if item['seed'] is not None:
            result = re.search(r'\d+', item['seed'])
            if result is not None:
                item['seed'] = int(result.group())

        # Convert the rank field to an integer
        if item['rank'] is not None:
            try:
                item['rank'] = int(item['rank'])
            except ValueError:
                item['rank'] = None

        # Fix the round field by replacing 
        # multiple spaces with a single space
        if item['round'] is not None:
            result = re.sub(r'\s+', ' ', item['round']).strip()
            match result:
                case 'Quarterfinals Quarter':
                    item['round'] = 'Quarterfinals'
                case 'Semifinals Semi':
                    item['round'] = 'Semifinals'
                case 'Final F':
                    item['round'] = 'Finals'
                case 'Round of 16 R16':
                    item['round'] = 'Round of 16'
                case 'Round of 32 R32':
                    item['round'] = 'Round of 32'
                case 'Round of 64 R64':
                    item['round'] = 'Round of 64'
                case 'Round of 128 R128':
                    item['round'] = 'Round of 128'
                case _:
                    item['round'] = result

        # Add details for the country related to the opponent
        country_code: str = item['country']

        item['code-alpha-3'] = None
        item['continent'] = None
        item['subregion'] = None
        
        if country_code is not None:
            result = get_country(country_code)
            if result:
                # item['country'] = result[0]['name']
                item['code-alpha-3'] = result[0]['code-alpha-3']

                continent = get_country_by_continent(country_code)

                if continent:
                    item['continent'] = continent[0]['continent']
                    item['subregion'] = continent[0]['subcontinent']

                # item['country'] = result[0]['name']

    return data

In [363]:
async def write_to_file(data: list, file_path: pathlib.Path):
    new_file_path = corrected_path / (file_path.stem + '_corrected.json')
    with new_file_path.open('w') as f:
        json.dump(data, f, indent=4)

In [364]:
for index, item in enumerate(files):
    with item.open() as f:
        data = json.load(f)
        for tournament in data:
            corrected_data = correct_data(tournament)
            
        task = asyncio.create_task(write_to_file(data, item))
        await task